In [130]:
# importing libraries
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

In [131]:
np.random.seed(42)                                                                      # duplicating results
X,y=make_regression(n_samples=1000,n_features=5,random_state=42,noise=20)               # creating synthetic dataset
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)
ss=StandardScaler()                                                                     # scaling
X_train_scaled=ss.fit_transform(X_train)
X_test_scaled=ss.transform(X_test)


In [132]:
# the optimal parameters calculated by the scikit learn algorithm:
lr=LinearRegression()
lr.fit(X_train,y_train)
m_optimal=lr.coef_
b_optimal=lr.intercept_
y_pred_optimal=lr.predict(X_test)
score_optimal=root_mean_squared_error(y_pred_optimal,y_test)
print(f"Optimal parameters calculated by scikit learn : {m_optimal,b_optimal}")

Optimal parameters calculated by scikit learn : (array([27.19345416, 45.72121925, 16.34627591, 24.07334351, 19.08779291]), np.float64(-1.1544622085771485))


In [133]:
# Now lets define our own MLR :
class MLR():
    def __init__(self,alpha=0.001,epochs=1000):
        self.alpha=alpha
        self.epochs=epochs

    def dL_dm(self,X,y,m,b):
        '''this returns the gradient vector containing summation of partial derivatives/slope of loss function 
        at specific m and b for whole data set X, currently X is a matrix (MLR) and m is vector.
        X= input
        y= true value
        m= weight vector
        b= intercept'''

        slope=np.zeros(X.shape[1])                                                      # size=no of features
        for i in range(len(X)):                 
            slope+= 2*(y[i]-np.matmul(m,X[i])-b)*(-X[i])                                # vectorization 
        return slope                                                                    # gradient vector returned

    def dL_db(self,X,y,m,b):
        '''this returns summation of partial derivative/slope of loss fn for whole dataset at given m vector and b'''
        slope=0
        for i in range(len(X)):
            slope+= 2*(y[i]-np.dot(m,X[i])-b)*(-1)                                      
        return slope

    def fit(self,X,y):
        self.m_old=np.array([np.random.randint(100) for _ in range(X.shape[1])])
        self.b_old=np.random.randint(100)
        for iter in range(self.epochs):
            # updating weight vector 'm':
            self.m_new=self.m_old - (self.alpha * self.dL_dm(X,y,self.m_old,self.b_old))
            self.m_old=self.m_new
            # updating bias 'b'
            self.b_new=self.b_old - (self.alpha * self.dL_db(X,y,self.m_old,self.b_old))
            self.b_old=self.b_new

    def predict(self,X):
        '''this returns the predicted values for each data point X[i] at the calculated 'm' vector and b'''
        y_pred=[]
        for i in range(X.shape[0]):
            y_pred.append(np.dot(self.m_old,X[i]) + self.b_old)
        return y_pred


In [134]:
regressor=MLR()
regressor.fit(X_train,y_train)
y_pred=regressor.predict(X_test)
score=root_mean_squared_error(y_pred,y_test)
print(f"The Root Mean Squared Error for scikit learn regressor: {score_optimal}")
print(f"The Root Mean Squared Error for implemented regressor: {score}")

The Root Mean Squared Error for scikit learn regressor: 21.069878754391123
The Root Mean Squared Error for implemented regressor: 21.069878754391123


In [135]:
print(f"The coef of LR in scikit learn : m -> {m_optimal},b -> {b_optimal}.")
m_old=[float(regressor.m_old[i]) for i in range(X.shape[1])]
print(f"The coef of SLR made by me : m -> {m_old},b -> {regressor.b_old}.")

The coef of LR in scikit learn : m -> [27.19345416 45.72121925 16.34627591 24.07334351 19.08779291],b -> -1.1544622085771485.
The coef of SLR made by me : m -> [27.193454156468555, 45.7212192454634, 16.346275910318596, 24.073343514128542, 19.087792909453512],b -> -1.154462208577151.
